# Predicting particle orientation from position and aspect ratio (B.Tech project, IIT Roorkee)

**Task.** For elliptical-cylinder particles in a vibrated bed (DEM simulation), predict each particle's orientation angle (0-180 degrees) from its position `(x, y, z)` and its `aspect_ratio`, using a Random Forest.

**What this notebook does**
1. Loads the DEM particle CSVs and cleans them (same steps as the original BTP notebook).
2. Reproduces the original result: a **random 80/20 split** of all particle rows.
3. Reports the metrics under their correct names. The BTP report's "R² = 0.833" is the share of predictions within ±35% of the true angle, **not** an R². The R² of the same model is about 0.69.
4. Adds a **grouped split** (hold out whole CSV files) and a mean-predictor baseline, because particles in the same file are strongly correlated and a random split is probably optimistic.

**Data.** The simulation CSVs are not in this repository (about 3 million rows). Put them in `data/` (or set the environment variable `BTP_DATA_DIR`). Each file needs the columns `Points:0`, `Points:1`, `Points:2`, `aspect_ratio` and `Orientation`.

In [ ]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, GroupKFold

DATA_DIR = Path(os.environ.get("BTP_DATA_DIR", "../data"))
FEATURES = ["Points:0", "Points:1", "Points:2", "aspect_ratio"]
TARGET = "Orientation"
SELECTED_COLUMNS = FEATURES + [TARGET]
PERIOD = 180.0          # orientation is an axis angle in 0-180 degrees (0 and 180 are the same alignment)
SAMPLE_FRAC = float(os.environ.get("BTP_SAMPLE_FRAC", 1.0))   # set < 1 for a quick run
N_TREES = 50

## 1. Load and clean

In [ ]:
files = sorted(glob.glob(str(DATA_DIR / "*.csv")))
assert files, f"No CSV files found in {DATA_DIR.resolve()}"

frames = []
for f in files:
    d = pd.read_csv(f, usecols=SELECTED_COLUMNS)
    d["source_file"] = Path(f).name          # remembers which simulation file each row came from
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
print("Files:", len(files), "| rows:", len(df))
df.isna().sum()

In [ ]:
# Some cells are non-numeric (the original notebook coerced them to NaN and dropped the rows)
num_cols = SELECTED_COLUMNS
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
n_before = len(df)
df = df.dropna(subset=num_cols).reset_index(drop=True)
print(f"Dropped {n_before - len(df):,} rows with missing/non-numeric values; {len(df):,} rows remain")
if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=0).reset_index(drop=True)
    print("Using a sample of", len(df), "rows")
df[TARGET].describe()

## 2. Metrics, with the right names

* `r2` and `mae`: standard regression metrics.
* `circular_mae`: mean absolute error that treats 179 degrees and 1 degree as 2 degrees apart (the angle wraps at 180).
* `within_35pct`: the share of predictions with `|y - y_hat| / y <= 0.35`. This is the quantity the BTP report called "R²". It depends on the size of the true angle (an error of 5 degrees counts very differently at 10 degrees than at 150 degrees) and is undefined when the true angle is 0, so I treat it as a descriptive number only.
* `within_10deg`: the share of predictions within 10 degrees on the wrapped scale (the 10 degrees is just an example threshold).

In [ ]:
def circular_abs_error(y_true, y_pred, period=PERIOD):
    d = np.abs(np.asarray(y_true) - np.asarray(y_pred)) % period
    return np.minimum(d, period - d)

def within_relative_tolerance(y_true, y_pred, tol=0.35):
    # the original BTP definition (mislabelled "R2 with tolerance")
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    with np.errstate(divide="ignore", invalid="ignore"):
        err = np.abs((y_true - y_pred) / y_true)
    return float(np.mean(err <= tol))

def report(y_true, y_pred, label):
    ce = circular_abs_error(y_true, y_pred)
    out = {
        "split": label,
        "r2": r2_score(y_true, y_pred),
        "mae_deg": mean_absolute_error(y_true, y_pred),
        "circular_mae_deg": float(ce.mean()),
        "within_35pct": within_relative_tolerance(y_true, y_pred),
        "within_10deg": float((ce <= 10).mean()),
    }
    return out

## 3. Original setup: random 80/20 split of all rows
This reproduces the BTP experiment (`random_state=10`, 50 trees). The original run on 3,082,565 cleaned rows gave **R² = 0.6907** and **0.8333 within ±35%**.

In [ ]:
X, y, groups = df[FEATURES], df[TARGET], df["source_file"]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, random_state=10, test_size=0.2)

rf = RandomForestRegressor(n_estimators=N_TREES, random_state=0, n_jobs=-1)
rf.fit(x_tr, y_tr)
rows = [report(y_te, rf.predict(x_te), "random 80/20 (original) - Random Forest")]

base = DummyRegressor(strategy="mean").fit(x_tr, y_tr)
rows.append(report(y_te, base.predict(x_te), "random 80/20 - predict the training mean"))
pd.DataFrame(rows).round(4)

## 4. Held-out files (grouped split)
Rows from the same simulation file share a shape, a stage and a spatial layout, so a random split lets the model see near-copies of the test particles. Here whole files are held out with `GroupKFold`, so every test particle comes from a file the model has never seen. If a file only differs from the others in its aspect ratio or stage, this is the harder and more honest question. The score will probably be lower than in section 3.

In [ ]:
n_groups = groups.nunique()
print("Number of source files:", n_groups)
if n_groups >= 3:
    k = min(5, n_groups)
    gkf = GroupKFold(n_splits=k)
    all_rows = []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups), 1):
        m = RandomForestRegressor(n_estimators=N_TREES, random_state=0, n_jobs=-1).fit(X.iloc[tr], y.iloc[tr])
        b = DummyRegressor(strategy="mean").fit(X.iloc[tr], y.iloc[tr])
        r_m = report(y.iloc[te], m.predict(X.iloc[te]), f"fold {fold} - Random Forest")
        r_b = report(y.iloc[te], b.predict(X.iloc[te]), f"fold {fold} - mean baseline")
        all_rows += [r_m, r_b]
    res = pd.DataFrame(all_rows)
    res["model"] = res["split"].str.contains("Forest").map({True: "Random Forest", False: "mean baseline"})
    display(res.groupby("model")[["r2", "mae_deg", "circular_mae_deg", "within_35pct", "within_10deg"]].agg(["mean", "std"]).round(3))
else:
    print("Need at least 3 files for a grouped split; skipping.")

## 5. Which inputs does the forest use?

In [ ]:
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
ax = imp.plot.barh(figsize=(5, 3), color="#4c72b0")
ax.set_xlabel("Impurity-based importance (model from section 3)")
plt.tight_layout(); plt.show()
imp.round(3)

## 6. Notes and limitations
* The random-split numbers in section 3 are what the BTP report reported (under a wrong metric name). They are kept here for comparison, not as a claim of how well the model generalises.
* Impurity-based importances can favour continuous variables. A permutation importance on the held-out files would be a better check.
* The target is an angle with period 180 degrees. A model that predicts the angle through `(sin 2θ, cos 2θ)` would respect the wrap-around, and is a natural next step.
* The report states a maximum tree depth of 30; the original notebook used scikit-learn's default depth. This notebook follows the notebook.